# TeleRAG-Agent: Full Setup & UI Launch on Kaggle

**Before running anything:**
1. ✅ Set **Accelerator → GPU T4 x2** (right panel)
2. ✅ Toggle **Internet → ON** (right panel) 
3. ✅ Click **Add Input → Datasets** and attach your `telerag-qdrant-db` dataset
4. ✅ Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`
   - Go to **Add-ons → Secrets → New Secret → Name: HF_TOKEN, Value: your_hf_token**
   - This is required to download the gated LLaMA-3 model

**Run cells ONE BY ONE** — each cell shows exactly what is happening so you always know.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: DIAGNOSTICS — Run this first to see your environment
# ═══════════════════════════════════════════════════════════════
import os, sys, subprocess, torch

print("=" * 60)
print("ENVIRONMENT DIAGNOSTICS")
print("=" * 60)
print(f"Python:      {sys.version.split()[0]}")
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"GPU Memory:  {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

print("\n" + "=" * 60)
print("AVAILABLE INPUT DATASETS")
print("=" * 60)
input_dirs = os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else []
if input_dirs:
    for d in input_dirs:
        print(f"  /kaggle/input/{d}")
else:
    print("  ⚠️  NO DATASETS ATTACHED — attach telerag-qdrant-db via Add Input!")

print("\n" + "=" * 60)
print("HF CACHE & DISK STATUS")
print("=" * 60)
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
print(f"HF Cache path: {hf_cache}")
if os.path.exists(hf_cache):
    cached = os.listdir(hf_cache)
    print(f"Cached models: {cached if cached else 'EMPTY (models will be downloaded)'}")  
else:
    print("HF Cache: EMPTY (models will be downloaded from HuggingFace)")
    
disk = subprocess.check_output("df -h /", shell=True, text=True).strip().split('\n')[-1].split()
print(f"Disk: {disk[2]} used / {disk[1]} total ({disk[4]} full)")
print("Need at least 25GB free for LLM + embeddings + qdrant")

print("\n✅ Diagnostics complete. Read output above before proceeding to Cell 2.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: INSTALL DEPENDENCIES
# (Run after Cell 1 — takes 2-3 minutes)
# ═══════════════════════════════════════════════════════════════
import os, sys

print("Step 1/3: Pinning sentence-transformers to 2.7.0 (avoids torchcodec/FFmpeg crash)...")
# Kaggle pre-installs v3.x which pulls in torchcodec -> FFmpeg crash
ret = os.system("pip install 'sentence-transformers==2.7.0' -q 2>&1 | tail -5")
print(f"  Exit code: {ret} ({'OK' if ret==0 else 'ERROR'})")

print("\nStep 2/3: Installing LangGraph, Qdrant, and other core deps...")
ret = os.system("""
pip install \
    'langgraph>=0.2.0' \
    'langchain-core>=0.2.0' \
    'qdrant-client>=1.9.0' \
    'fastembed>=0.3.6' \
    'FlagEmbedding>=1.2.0' \
    'python-dotenv' 'pyyaml' 'networkx' 'tqdm' \
    -q 2>&1 | tail -5
""")
print(f"  Exit code: {ret} ({'OK' if ret==0 else 'ERROR'})")

print("\nStep 3/3: Installing Gradio (pinned range to avoid API breakage)...")
ret = os.system("pip install 'gradio>=4.0.0,<5.0.0' -q 2>&1 | tail -3")
print(f"  Exit code: {ret} ({'OK' if ret==0 else 'ERROR'})")

# Verify the critical package
import importlib
import sentence_transformers
importlib.reload(sentence_transformers)
print(f"\n✅ sentence-transformers version: {sentence_transformers.__version__}")
print("✅ Dependencies installed. Proceed to Cell 3.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: CLONE REPO + SETUP QDRANT DATABASE
# ═══════════════════════════════════════════════════════════════
import os, sys, shutil

# ── CONFIG: Update this to match your dataset path ────────────
# Run !ls /kaggle/input/ in Cell 1 to find the correct name
QDRANT_DATASET_PATH = "/kaggle/input/telerag-qdrant-db/qdrant_storage"
REPO_DIR = "/kaggle/working/TeleRAG-Agent"
# ─────────────────────────────────────────────────────────────

print("Step 1/2: Cloning repository...")
if not os.path.exists(REPO_DIR):
    ret = os.system(f"git clone https://github.com/imchaitanya0/TeleRAG-Agent.git {REPO_DIR}")
    print(f"  Clone: {'OK' if ret==0 else 'FAILED'}")
else:
    ret = os.system(f"cd {REPO_DIR} && git pull origin main")
    print(f"  Pull latest: {'OK' if ret==0 else 'FAILED'}")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"  Working dir: {os.getcwd()}")

print("\nStep 2/2: Setting up Qdrant database...")
dest = f"{REPO_DIR}/data/qdrant_storage"
os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

if os.path.exists(dest) and len(os.listdir(dest)) > 0:
    print(f"  ✅ Qdrant storage already present at {dest}")
elif os.path.exists(QDRANT_DATASET_PATH):
    print(f"  Copying from {QDRANT_DATASET_PATH} ...")
    os.system(f"cp -r {QDRANT_DATASET_PATH} {dest}")
    size = os.popen(f"du -sh {dest}").read().strip()
    print(f"  ✅ Qdrant storage ready. Size: {size}")
else:
    print(f"  ⚠️  WARNING: {QDRANT_DATASET_PATH} not found!")
    print(f"  Available datasets: {os.listdir('/kaggle/input')}")
    print("  Chat tab will fail. Fix path and rerun this cell.")

print("\n✅ Setup complete. Proceed to Cell 4 to pre-download the LLM.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: PRE-DOWNLOAD MODELS WITH PROGRESS
# This cell downloads the 16GB LLM + embedding model BEFORE
# launching the UI so you can see download progress clearly.
# Takes ~10-20 min on first run. Subsequent runs: ~30 seconds.
# ═══════════════════════════════════════════════════════════════
import os, sys, time, torch
from pathlib import Path

REPO_DIR = "/kaggle/working/TeleRAG-Agent"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Get HF Token from Kaggle Secrets ──────────────────────────
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print(f"✅ HF Token loaded from Kaggle Secrets (length: {len(hf_token)})")
except Exception as e:
    print(f"⚠️  No HF_TOKEN secret found: {e}")
    print("   You may get 401 errors downloading gated models like LLaMA-3.")
    print("   Add it via: Add-ons → Secrets → New Secret → Name: HF_TOKEN")

# ── Model IDs from our config ─────────────────────────────────
LLM_MODEL_ID   = "AliMaatouk/LLama-3-8B-Tele-it"  # ~16GB (4-bit = ~4GB on GPU)
EMBED_MODEL_ID = "BAAI/bge-large-en-v1.5"          # ~1.3GB
RERANK_MODEL_ID = "BAAI/bge-reranker-v2-m3"        # ~1.1GB

print("\n" + "=" * 60)
print(f"Models to download:")
print(f"  1. LLM:      {LLM_MODEL_ID}")
print(f"  2. Embedder: {EMBED_MODEL_ID}")
print(f"  3. Reranker: {RERANK_MODEL_ID}")
print("=" * 60)

# ── Download Embedding model (fast, ~1.3GB) ────────────────────
print("\n[1/3] Downloading embedding model (~1.3 GB)...")
t0 = time.time()
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_ID, token=hf_token)
# Quick warmup to confirm it works
test_emb = embedder.encode("test")
print(f"  ✅ Embedding model ready! ({time.time()-t0:.0f}s) | Dim={len(test_emb)}")
del embedder  # free memory before loading LLM

# ── Download Reranker model (fast, ~1.1GB) ─────────────────────
print("\n[2/3] Downloading reranker model (~1.1 GB)...")
t0 = time.time()
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(RERANK_MODEL_ID, token=hf_token)
test_score = reranker.predict([["query", "document"]])
print(f"  ✅ Reranker ready! ({time.time()-t0:.0f}s)")
del reranker

# ── Download LLM (slow: 16GB raw, ~4GB in 4-bit) ──────────────
print(f"\n[3/3] Downloading LLM tokenizer + weights ({LLM_MODEL_ID})...")
print("  This downloads ~16GB of model shards — expect 10-20 minutes.")
print("  You will see 'Loading checkpoint shards' progress below:\n")

t0 = time.time()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("  Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  ✅ Tokenizer ready ({time.time()-t0:.0f}s)")

print("  Loading model in 4-bit (downloads shards, shows progress)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
model.eval()
elapsed = time.time() - t0
gpu_mem = torch.cuda.memory_allocated() / 1e9
print(f"  ✅ LLM loaded! ({elapsed:.0f}s) | GPU memory used: {gpu_mem:.1f} GB")

# ── Quick sanity generation test ───────────────────────────────
print("\n[SANITY CHECK] Running one inference to confirm model works...")
prompt = "### Question:\nQuestion: What is 5G NR?\n\n### Answer:\n"
inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"  Model output: {response[:200]!r}")

# Clean up — the app.py will reload through the singleton loader
del model, tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("✅ ALL MODELS DOWNLOADED AND VERIFIED!")
print("Models are now cached in ~/.cache/huggingface/")
print("Proceed to Cell 5 to launch the Gradio UI.")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: LAUNCH GRADIO UI
# Run ONLY after Cell 4 completes successfully.
# The UI will start in ~30 seconds (models already cached).
# ═══════════════════════════════════════════════════════════════
import os, sys, subprocess

REPO_DIR = "/kaggle/working/TeleRAG-Agent"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Load HF token again for the subprocess environment
hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except:
    pass

# Set environment so app.py can find the token + use cached models
env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["TRANSFORMERS_OFFLINE"] = "0"  # Allow fetching if anything missed
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"  # Show download bars

print("Launching TeleRAG-Agent Gradio UI with streaming output...")
print("Look for the 'gradio.live' URL below.")
print("-" * 60)

# Stream the app output line-by-line so you see exactly what's happening
proc = subprocess.Popen(
    [sys.executable, "src/ui/app.py", "--share"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    # Print every line as it comes (no buffering)
    print(line, end="", flush=True)

proc.wait()
print(f"\nProcess ended with exit code: {proc.returncode}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: DIAGNOSTICS IF SOMETHING IS STUCK/FAILING
# Run this in a NEW cell while Cell 5 is still running
# ═══════════════════════════════════════════════════════════════
import subprocess, torch

print("=" * 60)
print("RUNTIME DIAGNOSTICS")
print("=" * 60)

print("\n[1] Is app.py running?")
print(subprocess.check_output("ps aux | grep 'app.py' | grep -v grep", shell=True, text=True))

print("[2] Active network connections (to HuggingFace = still downloading):")
print(subprocess.check_output("ss -tunp 2>/dev/null | grep python | head -5", shell=True, text=True) or "  None")

print("[3] GPU memory usage:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        used = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {used:.1f} GB used / {total:.1f} GB total")

print("\n[4] HuggingFace cache contents (models downloaded so far):")
print(subprocess.check_output("ls ~/.cache/huggingface/hub/ 2>/dev/null || echo 'empty'", shell=True, text=True))

print("[5] Disk usage:")
print(subprocess.check_output("df -h / | tail -1", shell=True, text=True))